<a href="https://colab.research.google.com/github/syncXL/wazobia/blob/main/notebooks/noise_and_rir.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import os
import zipfile

In [3]:
%%capture
!wget -P /content/openslr/28  https://openslr.trmal.net/resources/28/rirs_noises.zip

In [7]:
def extract_zip_file(zip_file_path, extract_dir):
    """
    Extracts the contents of a zip file to a specified directory.

    Args:
        zip_file_path (str): The path to the zip file.
        extract_dir (str): The directory where the contents should be extracted.
    """
    if not os.path.exists(extract_dir):
        os.makedirs(extract_dir)

    try:
        with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)
        print(f"Successfully extracted '{zip_file_path}' to '{extract_dir}'")
    except zipfile.BadZipFile:
        print(f"Error: '{zip_file_path}' is not a valid zip file or is corrupted.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")


In [8]:
extract_zip_file("/content/openslr/28/rirs_noises.zip", "openslr/rirs_noises")

Successfully extracted '/content/openslr/28/rirs_noises.zip' to 'openslr/rirs_noises'


In [13]:
import shlex
from pathlib import Path


def parse_flag_file(path, trailing_key="path", skip_blank=True):
    """
    Parse a text file where each line looks like:
        --key1 value1 --key2 value2 ... [trailing positional]

    Returns a list of dicts, one per line. Any trailing token that is not
    preceded by a --key is stored under `trailing_key`.
    A --flag with no value (followed by another --flag or end of line) becomes True.
    """
    records = []
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if not line and skip_blank:
            continue

        tokens = shlex.split(line)
        rec, i = {}, 0
        while i < len(tokens):
            tok = tokens[i]
            if tok.startswith("--"):
                key = tok[2:]
                nxt = tokens[i + 1] if i + 1 < len(tokens) else None
                if nxt is None or nxt.startswith("--"):
                    rec[key] = True
                    i += 1
                else:
                    rec[key] = nxt
                    i += 2
            else:
                # positional token with no key
                rec[trailing_key] = tok
                i += 1
        records.append(rec)
    return records

In [16]:
noise_list[-1]

{'noise-id': 'noise-free-sound-0842',
 'noise-type': 'point-source',
 'bg-fg-type': 'background',
 'path': 'RIRS_NOISES/pointsource_noises/noise-free-sound-0842.wav'}

In [70]:
noise_list = parse_flag_file("/content/openslr/rirs_noises/RIRS_NOISES/pointsource_noises/noise_list")
noise_list_rir = parse_flag_file("/content/openslr/rirs_noises/RIRS_NOISES/real_rirs_isotropic_noises/noise_list")
rir = parse_flag_file("/content/openslr/rirs_noises/RIRS_NOISES/real_rirs_isotropic_noises/rir_list")
simulated_rir_small = parse_flag_file("/content/openslr/rirs_noises/RIRS_NOISES/simulated_rirs/largeroom/rir_list")
simulated_rir_medium = parse_flag_file("/content/openslr/rirs_noises/RIRS_NOISES/simulated_rirs/mediumroom/rir_list")
simulated_rir_large = parse_flag_file("/content/openslr/rirs_noises/RIRS_NOISES/simulated_rirs/largeroom/rir_list")

In [71]:
noise_list_rir[0]

{'noise-id': '00001',
 'noise-type': 'isotropic',
 'room-linkage': 'RVB2014_largeroom1',
 'path': 'RIRS_NOISES/real_rirs_isotropic_noises/RVB2014_type1_noise_largeroom1_1.wav'}

In [72]:
rir[0]

{'rir-id': '00001',
 'room-id': 'RVB2014_largeroom1',
 'path': 'RIRS_NOISES/real_rirs_isotropic_noises/RVB2014_type1_rir_largeroom1_far_angla.wav'}

In [73]:
import soundfile as sf

def get_audio_info(path) -> tuple[int, int]:
    """Returns (sampling_rate, n_channels) without decoding the audio."""
    info = sf.info(path)
    return info.samplerate, info.channels

In [74]:
DIR = "/content/openslr/rirs_noises/"


In [75]:
def prep_audio(row, audio_key="path", add_dict=None):
    row[audio_key] = DIR + row[audio_key]
    if add_dict:
        row = row | add_dict
    return row

noise_list = [prep_audio(n) for n in noise_list]
noise_list_rir = [prep_audio(n) for n in noise_list_rir]
rir = [prep_audio(n) for n in rir]
simulated_rir_small = [prep_audio(n, add_dict= {"room_size" : "small"}) for n in simulated_rir_small]
simulated_rir_medium = [prep_audio(n, add_dict= {"room_size" : "medium"}) for n in simulated_rir_medium]
simulated_rir_large = [prep_audio(n, add_dict= {"room_size" : "large"}) for n in simulated_rir_large]

In [76]:
from datasets import load_dataset, DatasetDict, Dataset, Audio, concatenate_datasets

In [77]:


noise_list = Dataset.from_list(noise_list)
noise_list_rir = Dataset.from_list(noise_list_rir)
rir = Dataset.from_list(rir)
simulated_rir_small = Dataset.from_list(simulated_rir_small)
simulated_rir_medium = Dataset.from_list(simulated_rir_medium)
simulated_rir_large = Dataset.from_list(simulated_rir_large)

In [78]:
noise_list = noise_list.rename_column("path", "audio").cast_column("audio", Audio(sampling_rate=16_000))
noise_list_rir = noise_list_rir.rename_column("path", "audio").cast_column("audio", Audio(sampling_rate=16_000))
rir = rir.rename_column("path", "audio").cast_column("audio", Audio(sampling_rate=16_000))
simulated_rir_small = simulated_rir_small.rename_column("path", "audio").cast_column("audio", Audio(sampling_rate=16_000))
simulated_rir_medium = simulated_rir_medium.rename_column("path", "audio").cast_column("audio", Audio(sampling_rate=16_000))
simulated_rir_large = simulated_rir_large.rename_column("path", "audio").cast_column("audio", Audio(sampling_rate=16_000))



In [79]:
from IPython.display import Audio as AudDisp

In [82]:
AudDisp(noise_list[0]["audio"]["array"], rate=16000)

In [83]:
simulated_rir = concatenate_datasets([simulated_rir_small, simulated_rir_medium, simulated_rir_large])

In [84]:
full_data = DatasetDict({
    "noise_list" : noise_list,
    "noise_list_rir" : noise_list_rir,
    "rir" : rir,
    "simulated_rir" : simulated_rir
})

In [85]:
from huggingface_hub import notebook_login

In [86]:
notebook_login()

In [87]:
noise_list.push_to_hub("nolimitsxl/noise-and-rir",config_name="noise_list")
noise_list_rir.push_to_hub("nolimitsxl/noise-and-rir",config_name="noise_list_rir")
rir.push_to_hub("nolimitsxl/noise-and-rir",config_name="rir")
simulated_rir.push_to_hub("nolimitsxl/noise-and-rir",config_name="simulated_rir")

Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ? shards/s]

Map:   0%|          | 0/422 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Map:   0%|          | 0/7500 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/nolimitsxl/noise-and-rir/commit/8ac870a5c06cf8a74a21f3a71451438a5ee86bf8', commit_message='Upload dataset', commit_description='', oid='8ac870a5c06cf8a74a21f3a71451438a5ee86bf8', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/nolimitsxl/noise-and-rir', endpoint='https://huggingface.co', repo_type='dataset', repo_id='nolimitsxl/noise-and-rir'), pr_revision=None, pr_num=None)